In [ ]:
"""
DPatch Adversarial Patch Attack for YOLO-OBB on Kaggle
========================================================
Dataset  : pawanpauljay/dota-1-5  (via Kaggle input)
Model    : yolo11n-obb.pt  (Ultralytics; swap path for custom yolo26n-obb weights)
Task     : OBB (Oriented Bounding Box) detection on DOTA classes
Platform : Kaggle (GPU P100 / T4)

Metrics reported:
  - Baseline  : mAP50, mAP50-95, Precision, Recall  (clean images)
  - Attacked  : mAP50, mAP50-95, Precision, Recall  (patch-applied images)
  - Drop table printed at the end

Usage:
    Simply run all cells / run as script. No CLI args needed.
"""

# ─────────────────────────────────────────────────────────────
# 0.  INSTALL (Kaggle has most of these; pip only for safety)
# ─────────────────────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip("ultralytics", "pyyaml", "matplotlib")

# ─────────────────────────────────────────────────────────────
# 1.  IMPORTS
# ─────────────────────────────────────────────────────────────
import os, shutil, random, json, gc, warnings
import yaml, cv2, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# 2.  CONFIGURATION  ← edit here
# ─────────────────────────────────────────────────────────────
# Kaggle dataset path (pawanpauljay/dota-1-5)
SOURCE_IMAGES_DIR = "/kaggle/input/datasets/pawanpauljay/dota-1-5/dota/images"
SOURCE_LABELS_DIR = "/kaggle/input/datasets/pawanpauljay/dota-1-5/dota/dota_labels"

# Working directory (writable on Kaggle)
WORK_DIR        = "/kaggle/working/dpatch_yolo_obb"
CHECKPOINT_DIR  = os.path.join(WORK_DIR, "checkpoints")

# Model weights  ← swap to your yolo26n-obb.pt path if custom
YOLO_WEIGHTS = "yolo26n-obb.pt"   # will auto-download if not present

# Image / patch
IMG_SIZE   = 640
PATCH_SIZE = 150    # pixels (top-left placement)

# Training
TOTAL_ITERS        = 15000
EVAL_EVERY_K       = 3000      # evaluate mAP every N iters
WARMUP_ITERS       = 500
BATCH_SIZE         = 4
LEARNING_RATE      = 0.10
TV_LOSS_WEIGHT     = 0.002
EMA_ALPHA          = 0.05
GRAD_CLIP          = 1.0
MAX_JITTER         = 10        # px jitter for patch placement

# Subset sizes
NUM_TRAIN_IMAGES   = 1600
NUM_VAL_IMAGES     = 400
NUM_VIZ_IMAGES     = 5

# Eval thresholds
CONF_THRESH = 0.001   # low conf to match standard YOLO val
IOU_THRESH  = 0.5

SEED = 42

# ─────────────────────────────────────────────────────────────
# 3.  DOTA CLASS NAMES
# ─────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0: "plane",            1: "ship",
    2: "storage tank",     3: "baseball diamond",
    4: "tennis court",     5: "basketball court",
    6: "ground track field", 7: "harbor",
    8: "bridge",           9: "large vehicle",
   10: "small vehicle",   11: "helicopter",
   12: "roundabout",      13: "soccer ball field",
   14: "swimming pool",   15: "container crane",
}
NAME_TO_IDX = {v: k for k, v in CLASS_NAMES.items()}

# ─────────────────────────────────────────────────────────────
# 4.  HELPERS
# ─────────────────────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def convert_dota_to_yolo_obb(src_path: str, dst_path: str, 
                             img_width: int, img_height: int) -> bool:
    if not os.path.exists(src_path):
        return False
    lines_out = []
    
    with open(src_path, 'r') as fh:
        for line in fh:
            parts = line.strip().split()
            
            # 1. Skip headers or malformed lines
            # DOTA lines with boxes always have at least 9-10 elements
            if len(parts) < 9:
                continue
            
            # 2. Check if the first element is a number (to skip 'imagesource', etc.)
            if not parts[0].replace('.', '', 1).isdigit():
                continue

            try:
                coords = list(map(float, parts[:8]))
                # DOTA standard is often 'small-vehicle', your dict uses 'small vehicle'
                cls_name = parts[8].strip().lower().replace("-", " ")
                
                if cls_name not in NAME_TO_IDX:
                    continue
                
                cls_id = NAME_TO_IDX[cls_name]
                
                # Normalize coordinates
                norm = []
                for i in range(0, 8, 2):
                    norm.append(coords[i]     / img_width)
                    norm.append(coords[i + 1] / img_height)
                
                lines_out.append(
                    f"{cls_id} " + " ".join(f"{x:.6f}" for x in norm) + "\n"
                )
            except ValueError:
                continue
                
    if not lines_out:
        return False
        
    with open(dst_path, "w") as fh:
        fh.writelines(lines_out)
    return True

def apply_patch_tensor(image_tensor: torch.Tensor,
                       patch: torch.Tensor,
                       jitter: bool = True) -> torch.Tensor:
    """Stamp patch onto image batch (top-left + optional jitter)."""
    adv = image_tensor.clone()
    dx = random.randint(0, MAX_JITTER) if jitter else 0
    dy = random.randint(0, MAX_JITTER) if jitter else 0
    adv[:, :, dy:dy + PATCH_SIZE, dx:dx + PATCH_SIZE] = (
        patch.unsqueeze(0).expand(adv.shape[0], -1, -1, -1)
    )
    return adv


def total_variation_loss(patch: torch.Tensor) -> torch.Tensor:
    tv_h = torch.mean(torch.abs(patch[:, 1:, :] - patch[:, :-1, :]))
    tv_w = torch.mean(torch.abs(patch[:, :, 1:] - patch[:, :, :-1]))
    return (tv_h + tv_w) / 2.0


def bgr_to_tensor(bgr: np.ndarray, device: torch.device) -> torch.Tensor:
    """HWC BGR uint8 → CHW RGB float32 [0,1] tensor."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).to(device)


def patch_to_numpy_bgr(patch: torch.Tensor) -> np.ndarray:
    """CHW [0,1] tensor → HWC BGR uint8."""
    rgb = (patch.detach().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


# ─────────────────────────────────────────────────────────────
# 5.  LOSS FUNCTION (hook-based, direct gradient path)
# ─────────────────────────────────────────────────────────────

def build_loss_fn(train_model: YOLO, device: torch.device):
    """
    Hooks into the OBB detection head's cv2 (box) and cv3 (cls)
    branches.  Works for Ultralytics OBB heads (v8/v11 style).
    """
    nn_model = train_model.model

    # Locate the detection head (last module)
    head = nn_model.model[-1]

    if not (hasattr(head, "cv2") and hasattr(head, "cv3")):
        raise RuntimeError(
            "Cannot find cv2/cv3 on detection head. "
            "Check model architecture with: print(train_model.model.model[-1])"
        )

    _cls_feats: list  = []
    _box_feats: list  = []

    def _hook_cls(module, inp, out):
        _cls_feats.append(out)

    def _hook_box(module, inp, out):
        _box_feats.append(out)

    handles = []
    for branch in head.cv3:
        convs = [m for m in branch.modules() if isinstance(m, nn.Conv2d)]
        if convs:
            handles.append(convs[-1].register_forward_hook(_hook_cls))

    for branch in head.cv2:
        convs = [m for m in branch.modules() if isinstance(m, nn.Conv2d)]
        if convs:
            handles.append(convs[-1].register_forward_hook(_hook_box))

    def _forward(img: torch.Tensor):
        _cls_feats.clear()
        _box_feats.clear()
        nn_model.train()          # keeps BN in training mode so grads flow
        nn_model(img)             # forward only (we care about hooked outputs)
        return list(_cls_feats), list(_box_feats)

    def loss_fn(adv_img: torch.Tensor, patch: torch.Tensor) -> torch.Tensor:
        cls_outs, box_outs = _forward(adv_img)

        loss = torch.zeros(1, device=device, dtype=torch.float32).squeeze()

        # ── 1. Global class-confidence suppression ──────────────────────
        for lg in cls_outs:
            loss = loss + torch.mean(torch.sigmoid(lg) ** 2)

        # ── 2. Patch-region confusion (entropy maximisation) ────────────
        for lg in cls_outs:
            B, C, H, W = lg.shape
            px = max(1, int(PATCH_SIZE / IMG_SIZE * W))
            py = max(1, int(PATCH_SIZE / IMG_SIZE * H))
            region = lg[:, :, :py, :px]
            sig    = torch.sigmoid(region).clamp(1e-6, 1 - 1e-6)
            entropy = -(sig * torch.log(sig) + (1 - sig) * torch.log(1 - sig))
            loss = loss - 0.4 * entropy.mean()   # maximise entropy → confusion

        # ── 3. Box regression destruction ───────────────────────────────
        for bx in box_outs:
            loss = loss + 0.5 * torch.mean(torch.abs(bx))
            loss = loss + 0.3 * torch.var(bx)

        # ── 4. False-positive encouragement (suppress true probs) ───────
        for lg in cls_outs:
            loss = loss - 0.2 * torch.mean(torch.sigmoid(lg))

        # ── 5. Total variation (smooth patch) ───────────────────────────
        loss = loss + TV_LOSS_WEIGHT * total_variation_loss(patch)

        return loss

    return loss_fn, handles


# ─────────────────────────────────────────────────────────────
# 6.  CHECKPOINT
# ─────────────────────────────────────────────────────────────

def load_checkpoint(device: torch.device):
    ensure_dir(CHECKPOINT_DIR)
    sp = os.path.join(CHECKPOINT_DIR, "state.json")
    pp = os.path.join(CHECKPOINT_DIR, "patch.pt")
    if os.path.exists(sp) and os.path.exists(pp):
        with open(sp) as fh:
            state = json.load(fh)
        patch = torch.load(pp, map_location=device)
        patch = patch.clamp(0.0, 1.0).detach().clone().requires_grad_(True)
        print(f"[checkpoint] Resuming from iteration {state.get('iteration', 0)}")
        return patch, state.get("iteration", 0), state.get("history", [])
    print("[checkpoint] Fresh start — random patch init.")
    patch = torch.rand(
        (3, PATCH_SIZE, PATCH_SIZE), device=device, requires_grad=True
    )
    return patch, 0, []


def save_checkpoint(patch: torch.Tensor, iteration: int, history: list):
    torch.save(patch.detach().cpu(), os.path.join(CHECKPOINT_DIR, "patch.pt"))
    with open(os.path.join(CHECKPOINT_DIR, "state.json"), "w") as fh:
        json.dump({"iteration": iteration, "history": history}, fh)


# ─────────────────────────────────────────────────────────────
# 7.  BUILD DATASETS
# ─────────────────────────────────────────────────────────────

def build_datasets(all_valid: list):
    """
    Split files into train / val subsets and build clean val directory
    with YOLO-OBB labels.
    """
    random.shuffle(all_valid)
    val_files  = all_valid[:NUM_VAL_IMAGES]
    train_files = all_valid[NUM_VAL_IMAGES:NUM_VAL_IMAGES + NUM_TRAIN_IMAGES]

    clean_dir = os.path.join(WORK_DIR, "clean_val")
    if os.path.exists(clean_dir):
        print("[data] Clean val dir already exists — skipping rebuild.")
        return train_files, val_files, clean_dir

    print("[data] Building clean validation dataset …")
    ensure_dir(os.path.join(clean_dir, "images"))
    ensure_dir(os.path.join(clean_dir, "labels"))

    kept = 0
    for fname in val_files:
        src_img = os.path.join(SOURCE_IMAGES_DIR, fname)
        img_bgr = cv2.imread(src_img)
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        # resize to IMG_SIZE for consistency
        img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(os.path.join(clean_dir, "images", fname), img_bgr)

        lbl_src = os.path.join(SOURCE_LABELS_DIR,
                               fname.rsplit(".", 1)[0] + ".txt")
        lbl_dst = os.path.join(clean_dir, "labels",
                               fname.rsplit(".", 1)[0] + ".txt")
        ok = convert_dota_to_yolo_obb(lbl_src, lbl_dst, w, h)
        if not ok:
            # write empty label so YOLO val doesn't skip the image
            open(lbl_dst, "w").close()
        kept += 1

    print(f"[data] Saved {kept} clean val images.")
    return train_files, val_files, clean_dir


def make_yaml(base_dir: str, out_path: str):
    with open(out_path, "w") as fh:
        yaml.dump(
            {
                "path"  : os.path.abspath(base_dir),
                "train" : "images",
                "val"   : "images",
                "nc"    : len(CLASS_NAMES),
                "names" : list(CLASS_NAMES.values()),
            },
            fh,
        )


# ─────────────────────────────────────────────────────────────
# 8.  APPLY PATCH TO DISK (for evaluation)
# ─────────────────────────────────────────────────────────────

def create_attacked_val_dir(patch: torch.Tensor,
                             clean_dir: str,
                             val_files: list,
                             tag: str) -> str:
    """Write patched images + copy labels → return attacked dir path."""
    adv_dir = os.path.join(WORK_DIR, f"attacked_{tag}")
    if os.path.exists(adv_dir):
        shutil.rmtree(adv_dir)
    ensure_dir(os.path.join(adv_dir, "images"))
    ensure_dir(os.path.join(adv_dir, "labels"))

    patch_bgr = patch_to_numpy_bgr(patch)          # HWC BGR uint8

    for fname in val_files:
        # copy label
        lbl_name = fname.rsplit(".", 1)[0] + ".txt"
        src_lbl  = os.path.join(clean_dir, "labels", lbl_name)
        dst_lbl  = os.path.join(adv_dir,  "labels", lbl_name)
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)
        else:
            open(dst_lbl, "w").close()

        # apply patch to image
        src_img = os.path.join(clean_dir, "images", fname)
        img_bgr = cv2.imread(src_img)
        if img_bgr is None:
            continue
        img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))

        # random placement within top-left region
        max_x = IMG_SIZE - PATCH_SIZE
        max_y = IMG_SIZE - PATCH_SIZE
        rx = random.randint(0, max(0, max_x))
        ry = random.randint(0, max(0, max_y))
        img_bgr[ry:ry + PATCH_SIZE, rx:rx + PATCH_SIZE] = patch_bgr

        cv2.imwrite(os.path.join(adv_dir, "images", fname), img_bgr)

    return adv_dir


# ─────────────────────────────────────────────────────────────
# 9.  METRICS EVALUATION
# ─────────────────────────────────────────────────────────────

def run_val(eval_model: YOLO, data_yaml: str,
            device: torch.device, label: str) -> dict:
    """Run YOLO val and return dict with all four metrics."""
    eval_model.model.eval()
    metrics = eval_model.val(
        data    = data_yaml,
        task    = "obb",
        device  = device,
        imgsz   = IMG_SIZE,
        conf    = CONF_THRESH,
        iou     = IOU_THRESH,
        plots   = False,
        verbose = False,
        half    = False,
    )
    result = {
        "label"    : label,
        "mAP50"    : round(float(metrics.box.map50),  4),
        "mAP50_95" : round(float(metrics.box.map),    4),
        "precision": round(float(metrics.box.mp),     4),
        "recall"   : round(float(metrics.box.mr),     4),
    }
    _print_metrics(result)
    return result


def _print_metrics(m: dict):
    print(
        f"\n  ┌─ {m['label']}\n"
        f"  │  mAP50      : {m['mAP50']:.4f}\n"
        f"  │  mAP50-95   : {m['mAP50_95']:.4f}\n"
        f"  │  Precision  : {m['precision']:.4f}\n"
        f"  └─ Recall     : {m['recall']:.4f}\n"
    )


def print_comparison(baseline: dict, attacked: dict):
    keys = ["mAP50", "mAP50_95", "precision", "recall"]
    labels = ["mAP50", "mAP50-95", "Precision", "Recall"]
    print("\n" + "═" * 55)
    print(f"{'Metric':<14} {'Baseline':>10} {'Attacked':>10} {'Drop':>10}")
    print("─" * 55)
    for k, lbl in zip(keys, labels):
        b = baseline[k]
        a = attacked[k]
        d = a - b
        arrow = "▼" if d < 0 else ("▲" if d > 0 else " ")
        print(f"{lbl:<14} {b:>10.4f} {a:>10.4f} {arrow}{abs(d):>9.4f}")
    print("═" * 55)


# ─────────────────────────────────────────────────────────────
# 10.  VISUALISATION
# ─────────────────────────────────────────────────────────────

def visualise(eval_model: YOLO, clean_dir: str, adv_dir: str,
              val_files: list, iteration: int):
    subset = val_files[:NUM_VIZ_IMAGES]
    fig, axes = plt.subplots(len(subset), 2,
                             figsize=(14, 6 * len(subset)))
    if len(subset) == 1:
        axes = [axes]

    for i, fname in enumerate(subset):
        # clean
        ic = cv2.imread(os.path.join(clean_dir, "images", fname))
        if ic is not None:
            r = eval_model.predict(ic, imgsz=IMG_SIZE, verbose=False, task="obb")[0]
            axes[i][0].imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        axes[i][0].set_title("Clean"); axes[i][0].axis("off")

        # adversarial
        ia = cv2.imread(os.path.join(adv_dir, "images", fname))
        if ia is not None:
            r = eval_model.predict(ia, imgsz=IMG_SIZE, verbose=False, task="obb")[0]
            axes[i][1].imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        axes[i][1].set_title(f"DPatch @ iter {iteration}"); axes[i][1].axis("off")

    plt.tight_layout()
    save_path = os.path.join(WORK_DIR, f"vis_{iteration}.png")
    plt.savefig(save_path, dpi=120)
    plt.show()
    print(f"[vis] Saved → {save_path}")


# ─────────────────────────────────────────────────────────────
# 11.  GRADIENT SANITY CHECK
# ─────────────────────────────────────────────────────────────

def gradient_sanity_check(loss_fn, device: torch.device):
    print("\n── Gradient sanity check ──")
    tp = torch.rand((3, PATCH_SIZE, PATCH_SIZE), device=device,
                    requires_grad=True)
    ti = torch.rand((2, 3, IMG_SIZE, IMG_SIZE), device=device)
    adv = apply_patch_tensor(ti, tp, jitter=False)
    loss = loss_fn(adv, tp)
    print(f"   test loss : {loss.item():.5f}")
    loss.backward()
    gnorm = tp.grad.abs().mean().item() if tp.grad is not None else 0.0
    print(f"   grad mean abs : {gnorm:.7f}")
    if gnorm < 1e-7:
        raise RuntimeError(
            "Zero gradient! Hook registration failed. "
            "Inspect: print(train_model.model.model[-1])"
        )
    print("   ✓ gradient flow OK\n")
    del tp, ti, adv, loss
    torch.cuda.empty_cache()


# ─────────────────────────────────────────────────────────────
# 12.  MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────

def main():
    set_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[config] Device : {DEVICE}")
    print(f"[config] Patch  : {PATCH_SIZE}×{PATCH_SIZE}  |  Iters : {TOTAL_ITERS}")

    ensure_dir(WORK_DIR)

    # ── Load models ────────────────────────────────────────────
    print("\n[model] Loading weights …")
    train_model = YOLO(YOLO_WEIGHTS)
    eval_model  = YOLO(YOLO_WEIGHTS)

    train_model.model.to(DEVICE)
    eval_model.model.to(DEVICE)

    # Override class names
    for m in (train_model, eval_model):
        if hasattr(m.model, "names"):
            m.model.names = CLASS_NAMES

    # ── Find valid images ───────────────────────────────────────
    exts = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
    all_imgs = [
        f for f in os.listdir(SOURCE_IMAGES_DIR)
        if f.lower().endswith(exts)
        and os.path.exists(
            os.path.join(SOURCE_LABELS_DIR, f.rsplit(".", 1)[0] + ".txt")
        )
    ]
    if not all_imgs:
        raise RuntimeError(
            f"No images with labels found in {SOURCE_IMAGES_DIR}. "
            "Check Kaggle dataset path."
        )
    print(f"[data] Found {len(all_imgs)} labelled images.")

    # ── Build datasets ─────────────────────────────────────────
    train_files, val_files, clean_dir = build_datasets(all_imgs)
    print(f"[data] Train : {len(train_files)}  |  Val : {len(val_files)}")

    clean_yaml = os.path.join(WORK_DIR, "clean.yaml")
    make_yaml(clean_dir, clean_yaml)

    # ── Baseline metrics ───────────────────────────────────────
    print("\n[baseline] Computing baseline metrics on clean images …")
    baseline_metrics = run_val(eval_model, clean_yaml, DEVICE, "BASELINE (clean)")

    # ── Build loss & check gradients ───────────────────────────
    loss_fn, hook_handles = build_loss_fn(train_model, DEVICE)
    gradient_sanity_check(loss_fn, DEVICE)

    # ── Patch + optimiser ──────────────────────────────────────
    patch, start_iter, history = load_checkpoint(DEVICE)

    optimizer = torch.optim.Adam([patch], lr=LEARNING_RATE, amsgrad=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max  = max(1, TOTAL_ITERS - WARMUP_ITERS),
        eta_min= LEARNING_RATE * 0.05,
    )

    # ── Training ───────────────────────────────────────────────
    print("═" * 55)
    print("  DPatch training — YOLO-OBB  (DOTA)")
    print("═" * 55)

    loss_ema = None

    for it in range(start_iter, TOTAL_ITERS):

        optimizer.zero_grad(set_to_none=True)

        # sample batch
        fnames = random.choices(train_files, k=BATCH_SIZE)
        imgs   = []
        for fname in fnames:
            bgr = cv2.imread(os.path.join(SOURCE_IMAGES_DIR, fname))
            if bgr is None:
                continue
            bgr  = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE))
            imgs.append(bgr_to_tensor(bgr, DEVICE))

        if not imgs:
            continue

        img_t  = torch.stack(imgs)                       # NCHW [0,1]
        adv_t  = apply_patch_tensor(img_t, patch)

        loss   = loss_fn(adv_t, patch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_([patch], GRAD_CLIP)
        optimizer.step()

        with torch.no_grad():
            patch.data.clamp_(0.0, 1.0)

        if it >= WARMUP_ITERS:
            scheduler.step()

        lv = loss.item()
        loss_ema = lv if loss_ema is None else (1 - EMA_ALPHA) * loss_ema + EMA_ALPHA * lv

        if (it + 1) % 200 == 0:
            lr = optimizer.param_groups[0]["lr"]
            tv = total_variation_loss(patch).item()
            print(f"  [{it+1:6d}/{TOTAL_ITERS}]  "
                  f"loss(ema)={loss_ema:.5f}  lr={lr:.5f}  tv={tv:.4f}")

        if (it + 1) % 1000 == 0:
            save_checkpoint(patch, it + 1, history)
            torch.cuda.empty_cache(); gc.collect()

        # ── Periodic evaluation ────────────────────────────────
        if (it + 1) % EVAL_EVERY_K == 0 or (it + 1) == TOTAL_ITERS:
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache(); gc.collect()

            adv_dir  = create_attacked_val_dir(patch, clean_dir, val_files,
                                               tag=f"iter{it+1}")
            adv_yaml = os.path.join(WORK_DIR, f"attacked_iter{it+1}.yaml")
            make_yaml(adv_dir, adv_yaml)

            atk_m = run_val(eval_model, adv_yaml, DEVICE,
                            f"ATTACKED @ iter {it+1}")
            history.append({
                "iteration" : it + 1,
                "mAP50"     : atk_m["mAP50"],
                "mAP50_95"  : atk_m["mAP50_95"],
                "precision" : atk_m["precision"],
                "recall"    : atk_m["recall"],
            })
            save_checkpoint(patch, it + 1, history)

            # mid-training visualisation
            if (it + 1) < TOTAL_ITERS:
                visualise(eval_model, clean_dir, adv_dir, val_files, it + 1)

            # cleanup tmp dirs
            shutil.rmtree(adv_dir, ignore_errors=True)
            if os.path.exists(adv_yaml):
                os.remove(adv_yaml)

            torch.cuda.empty_cache(); gc.collect()

    # ── Clean up hooks ─────────────────────────────────────────
    for h in hook_handles:
        h.remove()

    print("\n[done] Training complete.")

    # ─────────────────────────────────────────────────────────
    # FINAL EVALUATION (full val set, fixed patch placement)
    # ─────────────────────────────────────────────────────────
    print("\n[final] Creating final attacked val set …")
    final_adv_dir  = create_attacked_val_dir(patch, clean_dir, val_files, "final")
    final_adv_yaml = os.path.join(WORK_DIR, "attacked_final.yaml")
    make_yaml(final_adv_dir, final_adv_yaml)

    # Save patch image
    patch_path = os.path.join(WORK_DIR, "dpatch_final.png")
    cv2.imwrite(patch_path, patch_to_numpy_bgr(patch))
    print(f"[patch] Saved → {patch_path}")

    print("\n[final] Running final evaluation …")
    final_metrics = run_val(eval_model, final_adv_yaml, DEVICE,
                            "ATTACKED (final)")

    # ── Summary table ─────────────────────────────────────────
    print_comparison(baseline_metrics, final_metrics)

    # ── Visualise final results ───────────────────────────────
    visualise(eval_model, clean_dir, final_adv_dir, val_files, TOTAL_ITERS)

    # ── Training curve ────────────────────────────────────────
    if history:
        iters = [h["iteration"] for h in history]
        maps  = [h["mAP50"]     for h in history]
        precs = [h["precision"] for h in history]
        recs  = [h["recall"]    for h in history]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].axhline(baseline_metrics["mAP50"], color="blue",
                        linestyle="--", label="Baseline mAP50")
        axes[0].plot(iters, maps, "r-o", label="DPatch mAP50")
        axes[0].set_title("mAP50 vs Iterations")
        axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("mAP50")
        axes[0].legend(); axes[0].grid(True)

        axes[1].axhline(baseline_metrics["precision"], color="blue",
                        linestyle="--", label="Baseline Precision")
        axes[1].plot(iters, precs, "g-o", label="DPatch Precision")
        axes[1].set_title("Precision vs Iterations")
        axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Precision")
        axes[1].legend(); axes[1].grid(True)

        axes[2].axhline(baseline_metrics["recall"], color="blue",
                        linestyle="--", label="Baseline Recall")
        axes[2].plot(iters, recs, "m-o", label="DPatch Recall")
        axes[2].set_title("Recall vs Iterations")
        axes[2].set_xlabel("Iteration"); axes[2].set_ylabel("Recall")
        axes[2].legend(); axes[2].grid(True)

        plt.suptitle("DPatch Attack Progress — YOLO-OBB (DOTA)", fontsize=14)
        plt.tight_layout()
        curve_path = os.path.join(WORK_DIR, "dpatch_progress.png")
        plt.savefig(curve_path, dpi=120)
        plt.show()
        print(f"[plot] Saved → {curve_path}")

    # ── Final summary ─────────────────────────────────────────
    print("\n[summary] All outputs in:", WORK_DIR)
    print("[summary] Done ✓")


# ─────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()